# Image Classification with Test-Time Adaptation Project

### Project Overview

*Test Time Adaptation* (TTA) is an advanced technique in artificial intelligence where a model is dynamically adapted or fine-tuned during the inference stage using the data it is currently processing. The primary goal of TTA methods is to enhance the model’s performance on new or unseen datasets. This is particularly valuable in addressing the challenges posed by domain shift, where test samples originate from a distribution that differs from the one used during training.

The focus of this project is to implement and explore a renowned TTA method known as **Marginal Entropy Minimization with One test point (MEMO)**. This technique aims to improve the model’s adaptability and accuracy on divergent data distributions by minimizing the predictive uncertainty for individual test instances.

Additionally, this work will introduce an innovative and curious approach to data augmentation within the TTA framework. Specifically, we will investigate the potential of using advanced mathematical tools for image decomposition. The technique of **Wavelet Decomposition** will be employed to break down an image into its sub-bands, providing a unique method of augmenting the data that could significantly enhance the model’s ability to generalize from limited information.

We will start by establishing a baseline performance using the **ResNet-50** model, pre-trained with **IMAGENET1K_V1** weights, on the challenging **ImageNet-A** test dataset. This initial step will set a reference point for assessing the improvements brought by the MEMO method and our novel data augmentation strategy using wavelet decomposition.

Through this systematic approach, we aim to demonstrate the effectiveness of TTA in managing domain shifts and enhancing model robustness, making it a compelling choice for real-world AI applications where the data environment is constantly evolving.

### Team

This project is conducted under the guidance of Dr. Elisa Ricci and Dr. Francesco Tonini. The team working on this initiative includes:

- Stefano Bertolasi
- Edoardo Fiorentino

## Preliminaries

In this section we will set up all the environment, including all the import from external packages and the parameters we used in whole notebook.

In [1]:
import torch
from torch import nn
import torch.optim as optim
import torchvision
import pandas as pd
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.datasets as D
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from PIL import Image
from io import BytesIO
from pathlib import Path
from torch.utils.data import Dataset
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms import v2
from torch.utils.tensorboard import SummaryWriter
import boto3
import matplotlib.pyplot as plt
import random
import pywt


import argparse

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

2025-01-30 18:44:44.143925: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-30 18:44:44.386958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-30 18:44:44.440428: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-30 18:44:44.457380: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-30 18:44:44.721105: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# Defining parameters of the NN

parser = argparse.ArgumentParser()
parser.add_argument('--data_path', default='imagenet-a')
parser.add_argument('--corruption', default='adversarial')
parser.add_argument('--batch_size', default=16, type=int)
parser.add_argument('--lr', default=0.005, type=float)
parser.add_argument('--Niter', default=1, type=int, help='Number of iterations updating network parameters with entropy loss')
parser.add_argument('--M', default=8, type=int, help='Number of augmentation per iteration')
parser.add_argument('--optimizer', default='sgd', type=str)
            

args = parser.parse_args("")

## Dataset and Dataloading

The **ImageNet-A** dataset is a specialized subset of the well-known ImageNet database, which is extensively used in computer vision to train and test image recognition models. ImageNet-A was developed to assess the robustness of these models by exposing them to more challenging scenarios. It consists of real-world images that commonly used neural networks, initially trained on the standard ImageNet, often fail to recognize correctly. The primary aim of ImageNet-A is to serve as a benchmark for evaluating how well models perform when faced with difficult or adversarial examples.

One of the critical aspects of ImageNet-A is its class selection. Out of the 1,000 categories in the standard ImageNet, only 200 were chosen for ImageNet-A. This decision was based on targeting egregious errors that reveal significant weaknesses in classification systems. For example, confusing a domestic animal with a completely unrelated object dramatically highlights a model’s limitations. The chosen categories are designed to avoid rare, outdated, overly broad, or ambiguous classes, making the dataset a rigorous tool for pushing the limits of current image classification technologies.

ImageNet-A plays a crucial role in the development of computer vision models as it challenges them to improve and adapt to real-world conditions. As these models are increasingly deployed in various applications, ensuring they can withstand and correctly interpret adversarial or complex inputs is vital for their reliability and effectiveness.

Here we can see a picture from the original paper showing some image examples in ImageNet-A and their missclassification.

<img src="https://github.com/hendrycks/natural-adv-examples/blob/master/examples.png?raw=true" width="500" height="500">

Let us firstly create a mask array to separate images in ImageNet-A from others. Specifically, this mask array is used to identify and filter out the images that correspond to the selected 200 challenging classes from the full ImageNet dataset. By applying this mask, we can focus our analysis and model testing solely on those images that are known to cause classification errors, ensuring that our evaluations are concentrated on improving model performance under the most demanding conditions.

In the code below, <code style="background-color:lightgrey; padding:2px 5px; border-radius:5px;">all_wnids</code> contains the WNIDs for all classes in the full ImageNet dataset, and <code style="background-color:lightgrey; padding:2px 5px; border-radius:5px;">imagenet_a_wnids</code> is the list of WNIDs for the 200 challenging classes selected for ImageNet-A. The mask is created by checking each WNID in <code style="background-color:lightgrey; padding:2px 5px; border-radius:5px;">all_wnids</code> to see if it exists in <code style="background-color:lightgrey; padding:2px 5px; border-radius:5px;">imagenet_a_wnids</code>.





In [3]:
all_wnids = ['n01440764', 'n01443537', 'n01484850', 'n01491361', 'n01494475', 'n01496331', 'n01498041', 'n01514668', 'n01514859', 'n01518878', 'n01530575', 'n01531178', 'n01532829', 'n01534433', 'n01537544', 'n01558993', 'n01560419', 'n01580077', 'n01582220', 'n01592084', 'n01601694', 'n01608432', 'n01614925', 'n01616318', 'n01622779', 'n01629819', 'n01630670', 'n01631663', 'n01632458', 'n01632777', 'n01641577', 'n01644373', 'n01644900', 'n01664065', 'n01665541', 'n01667114', 'n01667778', 'n01669191', 'n01675722', 'n01677366', 'n01682714', 'n01685808', 'n01687978', 'n01688243', 'n01689811', 'n01692333', 'n01693334', 'n01694178', 'n01695060', 'n01697457', 'n01698640', 'n01704323', 'n01728572', 'n01728920', 'n01729322', 'n01729977', 'n01734418', 'n01735189', 'n01737021', 'n01739381', 'n01740131', 'n01742172', 'n01744401', 'n01748264', 'n01749939', 'n01751748', 'n01753488', 'n01755581', 'n01756291', 'n01768244', 'n01770081', 'n01770393', 'n01773157', 'n01773549', 'n01773797', 'n01774384', 'n01774750', 'n01775062', 'n01776313', 'n01784675', 'n01795545', 'n01796340', 'n01797886', 'n01798484', 'n01806143', 'n01806567', 'n01807496', 'n01817953', 'n01818515', 'n01819313', 'n01820546', 'n01824575', 'n01828970', 'n01829413', 'n01833805', 'n01843065', 'n01843383', 'n01847000', 'n01855032', 'n01855672', 'n01860187', 'n01871265', 'n01872401', 'n01873310', 'n01877812', 'n01882714', 'n01883070', 'n01910747', 'n01914609', 'n01917289', 'n01924916', 'n01930112', 'n01943899', 'n01944390', 'n01945685', 'n01950731', 'n01955084', 'n01968897', 'n01978287', 'n01978455', 'n01980166', 'n01981276', 'n01983481', 'n01984695', 'n01985128', 'n01986214', 'n01990800', 'n02002556', 'n02002724', 'n02006656', 'n02007558', 'n02009229', 'n02009912', 'n02011460', 'n02012849', 'n02013706', 'n02017213', 'n02018207', 'n02018795', 'n02025239', 'n02027492', 'n02028035', 'n02033041', 'n02037110', 'n02051845', 'n02056570', 'n02058221', 'n02066245', 'n02071294', 'n02074367', 'n02077923', 'n02085620', 'n02085782', 'n02085936', 'n02086079', 'n02086240', 'n02086646', 'n02086910', 'n02087046', 'n02087394', 'n02088094', 'n02088238', 'n02088364', 'n02088466', 'n02088632', 'n02089078', 'n02089867', 'n02089973', 'n02090379', 'n02090622', 'n02090721', 'n02091032', 'n02091134', 'n02091244', 'n02091467', 'n02091635', 'n02091831', 'n02092002', 'n02092339', 'n02093256', 'n02093428', 'n02093647', 'n02093754', 'n02093859', 'n02093991', 'n02094114', 'n02094258', 'n02094433', 'n02095314', 'n02095570', 'n02095889', 'n02096051', 'n02096177', 'n02096294', 'n02096437', 'n02096585', 'n02097047', 'n02097130', 'n02097209', 'n02097298', 'n02097474', 'n02097658', 'n02098105', 'n02098286', 'n02098413', 'n02099267', 'n02099429', 'n02099601', 'n02099712', 'n02099849', 'n02100236', 'n02100583', 'n02100735', 'n02100877', 'n02101006', 'n02101388', 'n02101556', 'n02102040', 'n02102177', 'n02102318', 'n02102480', 'n02102973', 'n02104029', 'n02104365', 'n02105056', 'n02105162', 'n02105251', 'n02105412', 'n02105505', 'n02105641', 'n02105855', 'n02106030', 'n02106166', 'n02106382', 'n02106550', 'n02106662', 'n02107142', 'n02107312', 'n02107574', 'n02107683', 'n02107908', 'n02108000', 'n02108089', 'n02108422', 'n02108551', 'n02108915', 'n02109047', 'n02109525', 'n02109961', 'n02110063', 'n02110185', 'n02110341', 'n02110627', 'n02110806', 'n02110958', 'n02111129', 'n02111277', 'n02111500', 'n02111889', 'n02112018', 'n02112137', 'n02112350', 'n02112706', 'n02113023', 'n02113186', 'n02113624', 'n02113712', 'n02113799', 'n02113978', 'n02114367', 'n02114548', 'n02114712', 'n02114855', 'n02115641', 'n02115913', 'n02116738', 'n02117135', 'n02119022', 'n02119789', 'n02120079', 'n02120505', 'n02123045', 'n02123159', 'n02123394', 'n02123597', 'n02124075', 'n02125311', 'n02127052', 'n02128385', 'n02128757', 'n02128925', 'n02129165', 'n02129604', 'n02130308', 'n02132136', 'n02133161', 'n02134084', 'n02134418', 'n02137549', 'n02138441', 'n02165105', 'n02165456', 'n02167151', 'n02168699', 'n02169497', 'n02172182', 'n02174001', 'n02177972', 'n02190166', 'n02206856', 'n02219486', 'n02226429', 'n02229544', 'n02231487', 'n02233338', 'n02236044', 'n02256656', 'n02259212', 'n02264363', 'n02268443', 'n02268853', 'n02276258', 'n02277742', 'n02279972', 'n02280649', 'n02281406', 'n02281787', 'n02317335', 'n02319095', 'n02321529', 'n02325366', 'n02326432', 'n02328150', 'n02342885', 'n02346627', 'n02356798', 'n02361337', 'n02363005', 'n02364673', 'n02389026', 'n02391049', 'n02395406', 'n02396427', 'n02397096', 'n02398521', 'n02403003', 'n02408429', 'n02410509', 'n02412080', 'n02415577', 'n02417914', 'n02422106', 'n02422699', 'n02423022', 'n02437312', 'n02437616', 'n02441942', 'n02442845', 'n02443114', 'n02443484', 'n02444819', 'n02445715', 'n02447366', 'n02454379', 'n02457408', 'n02480495', 'n02480855', 'n02481823', 'n02483362', 'n02483708', 'n02484975', 'n02486261', 'n02486410', 'n02487347', 'n02488291', 'n02488702', 'n02489166', 'n02490219', 'n02492035', 'n02492660', 'n02493509', 'n02493793', 'n02494079', 'n02497673', 'n02500267', 'n02504013', 'n02504458', 'n02509815', 'n02510455', 'n02514041', 'n02526121', 'n02536864', 'n02606052', 'n02607072', 'n02640242', 'n02641379', 'n02643566', 'n02655020', 'n02666196', 'n02667093', 'n02669723', 'n02672831', 'n02676566', 'n02687172', 'n02690373', 'n02692877', 'n02699494', 'n02701002', 'n02704792', 'n02708093', 'n02727426', 'n02730930', 'n02747177', 'n02749479', 'n02769748', 'n02776631', 'n02777292', 'n02782093', 'n02783161', 'n02786058', 'n02787622', 'n02788148', 'n02790996', 'n02791124', 'n02791270', 'n02793495', 'n02794156', 'n02795169', 'n02797295', 'n02799071', 'n02802426', 'n02804414', 'n02804610', 'n02807133', 'n02808304', 'n02808440', 'n02814533', 'n02814860', 'n02815834', 'n02817516', 'n02823428', 'n02823750', 'n02825657', 'n02834397', 'n02835271', 'n02837789', 'n02840245', 'n02841315', 'n02843684', 'n02859443', 'n02860847', 'n02865351', 'n02869837', 'n02870880', 'n02871525', 'n02877765', 'n02879718', 'n02883205', 'n02892201', 'n02892767', 'n02894605', 'n02895154', 'n02906734', 'n02909870', 'n02910353', 'n02916936', 'n02917067', 'n02927161', 'n02930766', 'n02939185', 'n02948072', 'n02950826', 'n02951358', 'n02951585', 'n02963159', 'n02965783', 'n02966193', 'n02966687', 'n02971356', 'n02974003', 'n02977058', 'n02978881', 'n02979186', 'n02980441', 'n02981792', 'n02988304', 'n02992211', 'n02992529', 'n02999410', 'n03000134', 'n03000247', 'n03000684', 'n03014705', 'n03016953', 'n03017168', 'n03018349', 'n03026506', 'n03028079', 'n03032252', 'n03041632', 'n03042490', 'n03045698', 'n03047690', 'n03062245', 'n03063599', 'n03063689', 'n03065424', 'n03075370', 'n03085013', 'n03089624', 'n03095699', 'n03100240', 'n03109150', 'n03110669', 'n03124043', 'n03124170', 'n03125729', 'n03126707', 'n03127747', 'n03127925', 'n03131574', 'n03133878', 'n03134739', 'n03141823', 'n03146219', 'n03160309', 'n03179701', 'n03180011', 'n03187595', 'n03188531', 'n03196217', 'n03197337', 'n03201208', 'n03207743', 'n03207941', 'n03208938', 'n03216828', 'n03218198', 'n03220513', 'n03223299', 'n03240683', 'n03249569', 'n03250847', 'n03255030', 'n03259280', 'n03271574', 'n03272010', 'n03272562', 'n03290653', 'n03291819', 'n03297495', 'n03314780', 'n03325584', 'n03337140', 'n03344393', 'n03345487', 'n03347037', 'n03355925', 'n03372029', 'n03376595', 'n03379051', 'n03384352', 'n03388043', 'n03388183', 'n03388549', 'n03393912', 'n03394916', 'n03400231', 'n03404251', 'n03417042', 'n03424325', 'n03425413', 'n03443371', 'n03444034', 'n03445777', 'n03445924', 'n03447447', 'n03447721', 'n03450230', 'n03452741', 'n03457902', 'n03459775', 'n03461385', 'n03467068', 'n03476684', 'n03476991', 'n03478589', 'n03481172', 'n03482405', 'n03483316', 'n03485407', 'n03485794', 'n03492542', 'n03494278', 'n03495258', 'n03496892', 'n03498962', 'n03527444', 'n03529860', 'n03530642', 'n03532672', 'n03534580', 'n03535780', 'n03538406', 'n03544143', 'n03584254', 'n03584829', 'n03590841', 'n03594734', 'n03594945', 'n03595614', 'n03598930', 'n03599486', 'n03602883', 'n03617480', 'n03623198', 'n03627232', 'n03630383', 'n03633091', 'n03637318', 'n03642806', 'n03649909', 'n03657121', 'n03658185', 'n03661043', 'n03662601', 'n03666591', 'n03670208', 'n03673027', 'n03676483', 'n03680355', 'n03690938', 'n03691459', 'n03692522', 'n03697007', 'n03706229', 'n03709823', 'n03710193', 'n03710637', 'n03710721', 'n03717622', 'n03720891', 'n03721384', 'n03724870', 'n03729826', 'n03733131', 'n03733281', 'n03733805', 'n03742115', 'n03743016', 'n03759954', 'n03761084', 'n03763968', 'n03764736', 'n03769881', 'n03770439', 'n03770679', 'n03773504', 'n03775071', 'n03775546', 'n03776460', 'n03777568', 'n03777754', 'n03781244', 'n03782006', 'n03785016', 'n03786901', 'n03787032', 'n03788195', 'n03788365', 'n03791053', 'n03792782', 'n03792972', 'n03793489', 'n03794056', 'n03796401', 'n03803284', 'n03804744', 'n03814639', 'n03814906', 'n03825788', 'n03832673', 'n03837869', 'n03838899', 'n03840681', 'n03841143', 'n03843555', 'n03854065', 'n03857828', 'n03866082', 'n03868242', 'n03868863', 'n03871628', 'n03873416', 'n03874293', 'n03874599', 'n03876231', 'n03877472', 'n03877845', 'n03884397', 'n03887697', 'n03888257', 'n03888605', 'n03891251', 'n03891332', 'n03895866', 'n03899768', 'n03902125', 'n03903868', 'n03908618', 'n03908714', 'n03916031', 'n03920288', 'n03924679', 'n03929660', 'n03929855', 'n03930313', 'n03930630', 'n03933933', 'n03935335', 'n03937543', 'n03938244', 'n03942813', 'n03944341', 'n03947888', 'n03950228', 'n03954731', 'n03956157', 'n03958227', 'n03961711', 'n03967562', 'n03970156', 'n03976467', 'n03976657', 'n03977966', 'n03980874', 'n03982430', 'n03983396', 'n03991062', 'n03992509', 'n03995372', 'n03998194', 'n04004767', 'n04005630', 'n04008634', 'n04009552', 'n04019541', 'n04023962', 'n04026417', 'n04033901', 'n04033995', 'n04037443', 'n04039381', 'n04040759', 'n04041544', 'n04044716', 'n04049303', 'n04065272', 'n04067472', 'n04069434', 'n04070727', 'n04074963', 'n04081281', 'n04086273', 'n04090263', 'n04099969', 'n04111531', 'n04116512', 'n04118538', 'n04118776', 'n04120489', 'n04125021', 'n04127249', 'n04131690', 'n04133789', 'n04136333', 'n04141076', 'n04141327', 'n04141975', 'n04146614', 'n04147183', 'n04149813', 'n04152593', 'n04153751', 'n04154565', 'n04162706', 'n04179913', 'n04192698', 'n04200800', 'n04201297', 'n04204238', 'n04204347', 'n04208210', 'n04209133', 'n04209239', 'n04228054', 'n04229816', 'n04235860', 'n04238763', 'n04239074', 'n04243546', 'n04251144', 'n04252077', 'n04252225', 'n04254120', 'n04254680', 'n04254777', 'n04258138', 'n04259630', 'n04263257', 'n04264628', 'n04265275', 'n04266014', 'n04270147', 'n04273569', 'n04275548', 'n04277352', 'n04285008', 'n04286575', 'n04296562', 'n04310018', 'n04311004', 'n04311174', 'n04317175', 'n04325704', 'n04326547', 'n04328186', 'n04330267', 'n04332243', 'n04335435', 'n04336792', 'n04344873', 'n04346328', 'n04347754', 'n04350905', 'n04355338', 'n04355933', 'n04356056', 'n04357314', 'n04366367', 'n04367480', 'n04370456', 'n04371430', 'n04371774', 'n04372370', 'n04376876', 'n04380533', 'n04389033', 'n04392985', 'n04398044', 'n04399382', 'n04404412', 'n04409515', 'n04417672', 'n04418357', 'n04423845', 'n04428191', 'n04429376', 'n04435653', 'n04442312', 'n04443257', 'n04447861', 'n04456115', 'n04458633', 'n04461696', 'n04462240', 'n04465501', 'n04467665', 'n04476259', 'n04479046', 'n04482393', 'n04483307', 'n04485082', 'n04486054', 'n04487081', 'n04487394', 'n04493381', 'n04501370', 'n04505470', 'n04507155', 'n04509417', 'n04515003', 'n04517823', 'n04522168', 'n04523525', 'n04525038', 'n04525305', 'n04532106', 'n04532670', 'n04536866', 'n04540053', 'n04542943', 'n04548280', 'n04548362', 'n04550184', 'n04552348', 'n04553703', 'n04554684', 'n04557648', 'n04560804', 'n04562935', 'n04579145', 'n04579432', 'n04584207', 'n04589890', 'n04590129', 'n04591157', 'n04591713', 'n04592741', 'n04596742', 'n04597913', 'n04599235', 'n04604644', 'n04606251', 'n04612504', 'n04613696', 'n06359193', 'n06596364', 'n06785654', 'n06794110', 'n06874185', 'n07248320', 'n07565083', 'n07579787', 'n07583066', 'n07584110', 'n07590611', 'n07613480', 'n07614500', 'n07615774', 'n07684084', 'n07693725', 'n07695742', 'n07697313', 'n07697537', 'n07711569', 'n07714571', 'n07714990', 'n07715103', 'n07716358', 'n07716906', 'n07717410', 'n07717556', 'n07718472', 'n07718747', 'n07720875', 'n07730033', 'n07734744', 'n07742313', 'n07745940', 'n07747607', 'n07749582', 'n07753113', 'n07753275', 'n07753592', 'n07754684', 'n07760859', 'n07768694', 'n07802026', 'n07831146', 'n07836838', 'n07860988', 'n07871810', 'n07873807', 'n07875152', 'n07880968', 'n07892512', 'n07920052', 'n07930864', 'n07932039', 'n09193705', 'n09229709', 'n09246464', 'n09256479', 'n09288635', 'n09332890', 'n09399592', 'n09421951', 'n09428293', 'n09468604', 'n09472597', 'n09835506', 'n10148035', 'n10565667', 'n11879895', 'n11939491', 'n12057211', 'n12144580', 'n12267677', 'n12620546', 'n12768682', 'n12985857', 'n12998815', 'n13037406', 'n13040303', 'n13044778', 'n13052670', 'n13054560', 'n13133613', 'n15075141']
imagenet_a_wnids = ['n01498041', 'n01531178', 'n01534433', 'n01558993', 'n01580077', 'n01614925', 'n01616318', 'n01631663', 'n01641577', 'n01669191', 'n01677366', 'n01687978', 'n01694178', 'n01698640', 'n01735189', 'n01770081', 'n01770393', 'n01774750', 'n01784675', 'n01819313', 'n01820546', 'n01833805', 'n01843383', 'n01847000', 'n01855672', 'n01882714', 'n01910747', 'n01914609', 'n01924916', 'n01944390', 'n01985128', 'n01986214', 'n02007558', 'n02009912', 'n02037110', 'n02051845', 'n02077923', 'n02085620', 'n02099601', 'n02106550', 'n02106662', 'n02110958', 'n02119022', 'n02123394', 'n02127052', 'n02129165', 'n02133161', 'n02137549', 'n02165456', 'n02174001', 'n02177972', 'n02190166', 'n02206856', 'n02219486', 'n02226429', 'n02231487', 'n02233338', 'n02236044', 'n02259212', 'n02268443', 'n02279972', 'n02280649', 'n02281787', 'n02317335', 'n02325366', 'n02346627', 'n02356798', 'n02361337', 'n02410509', 'n02445715', 'n02454379', 'n02486410', 'n02492035', 'n02504458', 'n02655020', 'n02669723', 'n02672831', 'n02676566', 'n02690373', 'n02701002', 'n02730930', 'n02777292', 'n02782093', 'n02787622', 'n02793495', 'n02797295', 'n02802426', 'n02814860', 'n02815834', 'n02837789', 'n02879718', 'n02883205', 'n02895154', 'n02906734', 'n02948072', 'n02951358', 'n02980441', 'n02992211', 'n02999410', 'n03014705', 'n03026506', 'n03124043', 'n03125729', 'n03187595', 'n03196217', 'n03223299', 'n03250847', 'n03255030', 'n03291819', 'n03325584', 'n03355925', 'n03384352', 'n03388043', 'n03417042', 'n03443371', 'n03444034', 'n03445924', 'n03452741', 'n03483316', 'n03584829', 'n03590841', 'n03594945', 'n03617480', 'n03666591', 'n03670208', 'n03717622', 'n03720891', 'n03721384', 'n03724870', 'n03775071', 'n03788195', 'n03804744', 'n03837869', 'n03840681', 'n03854065', 'n03888257', 'n03891332', 'n03935335', 'n03982430', 'n04019541', 'n04033901', 'n04039381', 'n04067472', 'n04086273', 'n04099969', 'n04118538', 'n04131690', 'n04133789', 'n04141076', 'n04146614', 'n04147183', 'n04179913', 'n04208210', 'n04235860', 'n04252077', 'n04252225', 'n04254120', 'n04270147', 'n04275548', 'n04310018', 'n04317175', 'n04344873', 'n04347754', 'n04355338', 'n04366367', 'n04376876', 'n04389033', 'n04399382', 'n04442312', 'n04456115', 'n04482393', 'n04507155', 'n04509417', 'n04532670', 'n04540053', 'n04554684', 'n04562935', 'n04591713', 'n04606251', 'n07583066', 'n07695742', 'n07697313', 'n07697537', 'n07714990', 'n07718472', 'n07720875', 'n07734744', 'n07749582', 'n07753592', 'n07760859', 'n07768694', 'n07831146', 'n09229709', 'n09246464', 'n09472597', 'n09835506', 'n11879895', 'n12057211', 'n12144580', 'n12267677']
imagenet_a_mask = [wnid in set(imagenet_a_wnids) for wnid in all_wnids]

To import the dataset we use the same function provided during lectures.

In [4]:
class S3ImageFolder(Dataset):
    def __init__(self, root, transform=None):
        self.s3_bucket = "deeplearning2024-datasets"
        self.s3_region = "eu-west-1"
        self.s3_client = boto3.client("s3", region_name=self.s3_region, verify=True)
        self.transform = transform

        # Get list of objects in the bucket
        response = self.s3_client.list_objects_v2(Bucket=self.s3_bucket, Prefix=root)
        objects = response.get("Contents", [])
        while response.get("NextContinuationToken"):
            response = self.s3_client.list_objects_v2(
                Bucket=self.s3_bucket,
                Prefix=root,
                ContinuationToken=response["NextContinuationToken"]
            )
            objects.extend(response.get("Contents", []))

        # Iterate and keep valid files only
        self.instances = []
        for ds_idx, item in enumerate(objects):
            key = item["Key"]
            path = Path(key)
            
            # Check if file is valid
            if path.suffix.lower() not in (".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"):
                continue

            # Get label
            label = path.parent.name

            # Keep track of valid instances
            self.instances.append((label, key))

        # Sort classes in alphabetical order (as in ImageFolder)
        self.classes = sorted(set(label for label, _ in self.instances))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        try:
            label, key = self.instances[idx]
            
            # Download image from S3
            # response = self.s3_client.get_object(Bucket=self.s3_bucket, Key=key)
            # img_bytes = response["Body"]._raw_stream.data

            img_bytes = BytesIO()
            response = self.s3_client.download_fileobj(Bucket=self.s3_bucket, Key=key, Fileobj=img_bytes)
            # img_bytes = response["Body"]._raw_stream.data
            
            # Open image with PIL
            img = Image.open(img_bytes).convert("RGB")

            # Apply transformations if any
            if self.transform is not None:
                img = self.transform(img)
        except Exception as e:
            raise RuntimeError(f"Error loading image at index {idx}: {str(e)}")

        return img, self.class_to_idx[label]

In [5]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transforms = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop((224, 224)),
    T.ToTensor(),
    T.Normalize(mean, std)
])

In the same way, we use the same function introduced during lectures to define our data loading utility. Before that, we defined the common transformations to use with ImageNet-A images and ResNet models, in particular we resized each image at 256 px, we crop the image to match the input size with the original ResNet-50 training dimension and we normalize it in order to align images with model expectations.

In [6]:
def prepare_data(args, use_transforms, shuffle=False):
    test_transforms_local = transforms if use_transforms else None
    if args.corruption in ['adversarial', 'imagenet_v2']:
        test_set = S3ImageFolder(root=args.data_path, transform=test_transforms_local)
    else:
        raise Exception('Datasets not found!')

    # DataLoader setup
    test_loader = torch.utils.data.DataLoader(
        test_set,
        batch_size=args.batch_size,
        shuffle=shuffle,
        pin_memory=True,
        num_workers=8
    )

    return test_set, test_loader

## Model Initialization

In this section we initialize the model, together with the optimizer for the future Marginal Entropy Minimization and the Test Loader

In [7]:
def initialize_resnet(weights='v2'):
    if weights == 'v1':
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(device)
    elif weights == 'v2':
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device)
    return resnet

In [8]:
test_set, test_loader = prepare_data(args, use_transforms=True, shuffle=False) 
model = initialize_resnet('v1')
if args.optimizer == 'sgd':
    optimizer = optim.SGD(model.parameters(), lr=args.lr)
elif args.optimizer == 'adamw':
    optimizer = optim.AdamW(model.parameters(), lr=args.lr)

/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


## Baseline ResNet-50 Test on ImageNet-A

In [37]:
def test(test_loader, model, device, args):
    """
    Test for baseline model

    Inputs:
    - test_loader (DataLoader): DataLoader providing batches of images and labels for testing.
    - model (torch.nn.Module): The neural network model to be evaluated.
    - device (torch.device or str): The computing device (CPU or GPU) where the model is run.
    
    Outputs:
    - final_accuracy_top1 (float): The final top-1 accuracy percentage across the test dataset.
    - final_accuracy_top5 (float): The final top-5 accuracy percentage across the test dataset.
    """

    # Initial print statement to indicate the start of the test
    print('Inside Test')
    
    # Set the model to evaluation mode (important for specific layers like Dropout and BatchNorm)
    model.eval()

    # Initialize counters for samples processed and cumulative accuracies
    samples = 0.0
    cumulative_accuracy_top1 = 0.0
    cumulative_accuracy_top5 = 0.0
    
    # Use torch.no_grad to disable gradient calculation, which is not needed during model evaluation
    with torch.no_grad():
        for i, (data_batch, labels_batch) in enumerate(test_loader):
            # Move data and labels to the specified device (CPU or GPU)
            data_batch, labels_batch = data_batch.to(device), labels_batch.to(device)

            # Get the output from the model
            output = model(data_batch)
            
            # If corruption is set to 'adversarial', apply the specified mask to the output
            if args.corruption == 'adversarial':
                output = output[:, imagenet_a_mask]

            # Calculate and accumulate top-1 accuracy
            _, predicted_top1 = output.max(1)
            cumulative_accuracy_top1 += predicted_top1.eq(labels_batch).sum().item()
            
            # Calculate and accumulate top-5 accuracy
            _, predicted_top5 = output.topk(5, dim=1)
            correct_top5 = predicted_top5.eq(labels_batch.view(-1, 1).expand_as(predicted_top5))
            cumulative_accuracy_top5 += correct_top5.sum().item()

            # Update the total number of samples processed
            samples += data_batch.size(0)

            # Optionally print the current accuracy every 10 batches (code commented out)
            if (i + 1) % 10 == 0:
                current_accuracy_top1 = cumulative_accuracy_top1 / samples * 100
                current_accuracy_top5 = cumulative_accuracy_top5 / samples * 100
                print(f'Batch {i + 1}, Current Accuracy Top-1: {current_accuracy_top1:.4f}')
                print(f'Batch {i + 1}, Current Accuracy Top-5: {current_accuracy_top5:.4f}')

    # Calculate the final accuracies for the entire test set
    final_accuracy_top1 = cumulative_accuracy_top1 / samples * 100
    final_accuracy_top5 = cumulative_accuracy_top5 / samples * 100
    
    # Return the computed top-1 and top-5 accuracies
    return final_accuracy_top1, final_accuracy_top5

In [38]:
final_accuracy_top1, final_accuracy_top5 = test(test_loader, model, device, args)
print(f'Final Accuracy Top-1: {final_accuracy_top1:.4f}')
print(f'Final Accuracy Top-5: {final_accuracy_top5:.4f}')

Inside Test


/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Batch 10, Current Accuracy Top-1: 2.5000
Batch 10, Current Accuracy Top-5: 18.1250
Batch 20, Current Accuracy Top-1: 1.2500
Batch 20, Current Accuracy Top-5: 9.6875
Batch 30, Current Accuracy Top-1: 0.8333
Batch 30, Current Accuracy Top-5: 8.3333
Batch 40, Current Accuracy Top-1: 0.6250
Batch 40, Current Accuracy Top-5: 9.6875
Batch 50, Current Accuracy Top-1: 0.5000
Batch 50, Current Accuracy Top-5: 10.0000
Batch 60, Current Accuracy Top-1: 0.5208
Batch 60, Current Accuracy Top-5: 10.5208
Batch 70, Current Accuracy Top-1: 0.5357
Batch 70, Current Accuracy Top-5: 11.5179
Batch 80, Current Accuracy Top-1: 0.4688
Batch 80, Current Accuracy Top-5: 11.3281
Batch 90, Current Accuracy Top-1: 0.4861
Batch 90, Current Accuracy Top-5: 12.0139
Batch 100, Current Accuracy Top-1: 0.4375
Batch 100, Current Accuracy Top-5: 11.6250
Batch 110, Current Accuracy Top-1: 0.3977
Batch 110, Current Accuracy Top-5: 11.3636
Batch 120, Current Accuracy Top-1: 0.3646
Batch 120, Current Accuracy Top-5: 11.7708
B

## MEMO - Marginal Entropy Minimization with One-Test-Point

In this section we will illustrate the base implementation MEMO, one of the most known methods for *test time robustness*. The core idea of this method is to, when faced with a test point, adapt the model by varying the augmentation of the test point and promote consistent predictions across these modifications, thereby adhering to the invariances inherent in the data augmentations. To do so the model aims to minimize the marginal entropy of the model’s predictions across the different augmented versions of the test point.
Below one can see the general scheme of MEMO.

<img src="https://ar5iv.labs.arxiv.org/html/2110.09506/assets/fig/intro.png" width="500" height="500">

The base implementation of MEMO follows what is written in section 3.1 of the original paper. In this framework, starting from a pretrained model $f_\theta$ and a set of augmentation functions $\mathcal{A}$, we sample $M$ augmentations randomly from $\mathcal{A}$ and we apply them to the single test-point $\mathbf{x}_0$ in order to produce a batch of $M$ augmented data $\mathbf{x}_1, \dots \mathbf{x}_M$. Let $p_\theta(c|\mathbf{x})$ be the output probability of class $c$ given the input $\mathbf{x}$ and model with parameters $\theta$.

We compute the probability distributions for every augmented image in the batch and call it $p_\theta(\cdot|\mathbf{x}_i)$. Then the model average distribution over augmented points is defined as 
$$\overline{p}_\theta(c|\mathbf{x}) = \sum_{i = 1}^M p_\theta(c|\mathbf{x}_i)$$

Then we compute the marginal entropy of the average distribution, where the marginal entropy of a discrete probability distribution is defined by 
$$
H(X) = -\sum_{x \in \mathcal{X}} p(x) \log p(x)
$$

where $ p(x) $ represents the probability of each discrete outcome $ x $, and the sum is taken over all possible outcomes in the sample space $ \mathcal{X} $.
Then using the entropy as a loss function we perform a single backpropagation, obtaining updated model parameters $\theta^\prime$. With the updated model we the compute the predicted class.


In the following we can see the implementation of the marginal entropy function. One can see a slightly different implementation of the previous formula, this is done in order to guarantee numerical stability in computing the logarithm. In addiction, we also need to pay attention to the base of the logarithm. In general when computing the entropy the logarithm with base $2$ is used, however, in our case, for simplicity, we decided to use the classical logarithm. This procedure will not effect our algorithm since the two only differs for a proportional constant.


In [9]:
def marginal_entropy(outputs):
    """
    Compute the marginal entropy of a batch of logits.

    This function calculates the marginal entropy across all samples in a batch for each class.
    Marginal entropy is used to quantify the uncertainty of the model's predictions across the batch.

    Parameters:
        outputs (torch.Tensor): A tensor of shape (batch_size, num_classes) that
                               contains the logits for each class of each example in the batch.

    Returns:
        float: The computed marginal entropy of the distribution.
    """

    # Normalize logits to prevent numerical instability
    z = outputs - outputs.logsumexp(dim=-1, keepdim=True)  # Log-Sum-Exp trick for normalization
    
    # Calculate the log of marginal probabilities for each class across all examples
    marginal_logp = z.logsumexp(dim=0) - torch.log(torch.tensor(z.shape[0], dtype=torch.float32))

    # Obtain the smallest representable number to prevent log underflow during computation
    min_real = torch.finfo(marginal_logp.dtype).min
    # Clamp the log probabilities to a minimum value for numerical stability
    clamped_log_probs = torch.clamp(marginal_logp, min=min_real)

    # Calculate the marginal entropy by combining the clamped log probabilities with their exponentials
    marginal_entropy = -(clamped_log_probs * torch.exp(clamped_log_probs)).sum()

    return marginal_entropy

In [10]:
augmix = T.Compose([
    v2.AugMix()
])

def augment_single_image(single_image, args):
    """
    Augments a single image multiple times and processes it through ResNet-50 to compute the average output distribution.

    Inputs:
    - single_image: (torch.Tensor)

    Output:
    - avg_out_distr: (torch.tensor)
    """

    # Apply the augmentation to the image 'args.M' times using list comprehension
    augmented_images = [augmix(single_image) for _ in range(args.M)]
    
    # Stack the augmented images into a single tensor and move it to GPU
    augmented_images = torch.stack(augmented_images, dim=0).cuda()

    # Pass the augmented images through the model to get the output
    output = model(augmented_images)
    
    # If the corruption type is 'adversarial', filter the output using the imagenet_a_mask
    if args.corruption == 'adversarial':
        output = output[:, imagenet_a_mask]
    
    # Compute the average output distribution across all augmented images
    avg_out_distr = torch.mean(output, dim=0)
    
    # Return the average output distribution
    return avg_out_distr

In [32]:
def test_with_memo(test_loader, model, device, args):
    """
    Test function
    
    Inputs:
    - test_loader (DataLoader): DataLoader containing the dataset to be tested, providing batches of images and labels.
    - model (torch.nn.Module): The neural network model to be evaluated.
    - device (torch.device or str): The computing device (CPU or GPU) where the model is run.
    
    Outputs:
    - final_accuracy_top1 (float): Final top-1 accuracy percentage across the test set.
    - final_accuracy_top5 (float): Final top-5 accuracy percentage across the test set.
    """
    
    # Initial print to indicate the start of the testing phase
    print('Inside Test')
    
    # Initialize counters for samples processed and cumulative accuracies for top-1 and top-5
    samples = 0.0
    cumulative_accuracy_top1 = 0.0
    cumulative_accuracy_top5 = 0.0

    # Store the default state of the model parameters to restore later
    default_params = {}
    for param in model.parameters():
        default_params[param] = param.clone()
    
    # Iterate over each batch of images and labels in the test loader
    for i, (image_batch, label_batch) in enumerate(test_loader):
        
        # Move the image and label batch to the specified device
        image_batch, label_batch = image_batch.to(device), label_batch.to(device)

        # Set model to evaluation mode
        model.eval()
        
        # Perform augmentation and update model parameters multiple times per image
        for _ in range(args.Niter):
            # Reset gradient information
            optimizer.zero_grad()
            
            # Augment each image in the batch and calculate the average output distribution
            avg_out_distr = [augment_single_image(single_image, args) for single_image in image_batch]
            avg_out_distr = torch.stack(avg_out_distr, dim=0)

            # Calculate the loss from the average output distributions
            loss = marginal_entropy(avg_out_distr)
            loss.backward()
            optimizer.step()

        # Disable gradient calculations for performance
        with torch.no_grad():
            # Forward pass to get model outputs
            output = model(image_batch)

            # Apply mask if corruption is adversarial
            if args.corruption == 'adversarial':
                output = output[:, imagenet_a_mask]

            # Calculate top-1 accuracy
            _, predicted_top1 = output.max(1)
            cumulative_accuracy_top1 += predicted_top1.eq(label_batch).sum().item()
        
            # Calculate top-5 accuracy
            _, predicted_top5 = output.topk(5, dim=1)
            correct_top5 = predicted_top5.eq(label_batch.view(-1, 1).expand_as(predicted_top5))
            cumulative_accuracy_top5 += correct_top5.sum().item()

            # Update total samples processed
            samples += image_batch.size(0)

            # Print current accuracy every 10 images
            if (i + 1) % 10 == 0:
                current_accuracy_top1 = cumulative_accuracy_top1 / samples * 100
                current_accuracy_top5 = cumulative_accuracy_top5 / samples * 100
                print(f'Image {i + 1}, Current Accuracy Top-1: {current_accuracy_top1:.4f}')
                print(f'Image {i + 1}, Current Accuracy Top-5: {current_accuracy_top5:.4f}')
            
        # Restore the original parameters after each batch to avoid parameter drift
        for param in model.parameters():
            param.data = default_params[param].clone()

    # Calculate and return the final accuracies for the entire test set
    final_accuracy_top1 = cumulative_accuracy_top1 / samples * 100
    final_accuracy_top5 = cumulative_accuracy_top5 / samples * 100
    
    return final_accuracy_top1, final_accuracy_top5

In [33]:
final_accuracy_top1, final_accuracy_top5 = test_with_memo(test_loader, model, device, args)
print(f'Final Accuracy Top-1: {final_accuracy_top1:.4f}')
print(f'Final Accuracy Top-5: {final_accuracy_top5:.4f}')

Inside Test
Image 10, Current Accuracy Top-1: 19.3750
Image 10, Current Accuracy Top-5: 45.6250
Image 20, Current Accuracy Top-1: 9.6875
Image 20, Current Accuracy Top-5: 24.6875
Image 30, Current Accuracy Top-1: 6.6667
Image 30, Current Accuracy Top-5: 18.9583
Image 40, Current Accuracy Top-1: 5.1562
Image 40, Current Accuracy Top-5: 20.9375
Image 50, Current Accuracy Top-1: 4.1250
Image 50, Current Accuracy Top-5: 20.6250
Image 60, Current Accuracy Top-1: 4.7917
Image 60, Current Accuracy Top-5: 20.4167
Image 70, Current Accuracy Top-1: 4.8214
Image 70, Current Accuracy Top-5: 22.1429
Image 80, Current Accuracy Top-1: 4.8438
Image 80, Current Accuracy Top-5: 22.6562
Image 90, Current Accuracy Top-1: 4.7222
Image 90, Current Accuracy Top-5: 25.2778
Image 100, Current Accuracy Top-1: 4.2500
Image 100, Current Accuracy Top-5: 24.0625
Image 110, Current Accuracy Top-1: 3.9205
Image 110, Current Accuracy Top-5: 23.9773
Image 120, Current Accuracy Top-1: 4.1146
Image 120, Current Accuracy 

## W-MEMO: Wavelet-based enhancement of MEMO

Wavelet transforms are mathematical tools that decompose a signal into a set of basis functions called wavelets, each with different frequency and location. Unlike traditional Fourier transforms, which represent a signal as a sum of sinusoidal functions, wavelet transforms use localized waveforms, making them more effective for analyzing non-stationary signals with discontinuities and sharp transitions.

In mathematical terms, the continuous wavelet transform $W_f(a, b)$ of a function $f(t)$ is defined as:
\begin{equation}
W_f(a, b) = \int_{-\infty}^{\infty} f(t) \frac{1}{\sqrt{a}} \psi\left(\frac{t-b}{a}\right) \, dt
\end{equation}
where $a$ is the scale parameter, $b$ is the translation parameter, and $\psi(t)$ is a mother wavelet. Below one can see the exaample of three important mother wavelets that we will use also in our experiments.

<html>
<body>
    <div style="display: flex;">
        <div style="margin-right: 20px;">
            <img src="https://www.researchgate.net/publication/251118123/figure/fig1/AS:871114821619712@1584701366728/Example-of-wavelet-first-derivative-of-Gaussian.png" alt="Gaussian Wavelet" style="width:200px;">
            <p>Meyer</p>
        </div>
        <div style="margin-right: 20px;">
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/0/0a/MorletWaveletMathematica.svg/800px-MorletWaveletMathematica.svg.png" alt="Morlet Wavelet" style="width:200px;">
            <p>Morlet</p>
        </div>
        <div>
            <img src="https://www.researchgate.net/publication/379137648/figure/fig1/AS:11431281232113597@1711605634483/Example-of-mother-wavelet-Daubechies-4.tif" alt="Daubechies Wavelet" style="width:200px;">
            <p>Daubechies 4</p>
        </div>
    </div>
</body>
</html>


For image processing, the concept of wavelet transform is extended to two dimensions to effectively handle the spatial structure of images. In particular, when dealing with images we will talk about Two-Dimensional-Discrete-Wavelet-Transforms (DWT). The DWT of an image is calculated by passing it through a series of filters.
Formally, the DWT of an image is calculated by passing it through a series of bidimensional filters. Initially, the image $ I(x, y) $ undergoes a convolution with a low-pass filter with impulse response $g$, which is performed in two dimensions:
$$
cA[x, y] = (I \ast g)[x, y] = \sum_{m=-\infty}^{\infty} \sum_{n=-\infty}^{\infty} I[m, n] g[x - m, y - n]
$$
where $y[x, y]$ represents the approximation coefficients at a lower level of detail. Concurrently, the image is also decomposed using a high-pass filter $h$, which provides the detail coefficients. This convolution is applied separately to capture horizontal, vertical, and diagonal details:
$$
cH[x, y] = (I \ast h_h)[x, y]
$$
$$
cV[x, y] = (I \ast h_v)[x, y]
$$
$$
cD[x, y] = (I \ast h_d)[x, y]
$$
where $cH$, $cV$, and $cD$ are high-pass filters configured to isolate details in the horizontal, vertical, and diagonal directions, respectively. These coefficients $(cH , cV , cD)$ along with the approximation coefficients $ cA $ complete the decomposition of the image.

This transformation yields a comprehensive set of coefficients representing the image at various levels of detail and orientation. In our implementation we will deal with these coeffiecients in order to enhance the augmentations of the image at a different level. 

In particular, in the following, the DWT of the images will be done with the help of the python library PyWavelets. The DWT of the image will be done, and using an augmentation function, we will manipulate horizontal, vertical and diagonal coefficients in order to augment the image in the frequency domain. At the end of this procedure, the IDWT, Inverse Discrete Wavelet Transforms will be applied to augmented coeeficients and the "original augmented" image will be recomposed. We also report on some test using a multilevel wavelet decomposition. This can be obtained using the DWT recursively on the approximated image cA. Below we report an example of decomposed image using the DWT.

<html>
<body>
    <div style="display: flex;">
        <div style="margin-right: 10px;">
            <img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*JdvV-CqBjdMul-td81YwaA.png" alt="cA" style="width:200px;">
            <p>cA</p>
        </div>
        <div style="margin-right: 10px;">
            <img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*8fw3OLDgauTyIGCSODM-1Q.png" alt="cH" style="width:200px;">
            <p>cH</p>
        </div>
        <div style="margin-right: 10px;">
            <img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*yR_erKnrRXJ0uVP63zoxpw.png" alt="cV" style="width:200px;">
            <p>cV</p>
        </div>
        <div>
            <img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*Jg_mZ3vEuTyYACFMvtpHYg.png" alt="cD" style="width:200px;">
            <p>cD</p>
        </div>
    </div>
</body>
</html>

In the following we report three functions to be used in three different tests:
- decompose_and_inverse: performs a single_level wavelet decomposition of the image, augments its details and returns back to the original image iin order to augment it with Augmix
- only_decompose: performs a single_level wavelet decomposition of the image and returns the approximated image cA in order to apply Augmix directly on it
- multilevel_decompose: applies a multilevel decomposition of the image augmenting the coefficients at each stage. It then returns back the recomposed image.

In [11]:
def augment_details(coeff, method, intensity=0.1):
    """
    Apply augmentation to wavelet detail coefficients. In particular we add Normal distributed
    noise to horizontal coefficient, we enhance vertical coefficient and we mask 
    diagonal components.
    
    Inputs:
        coeff (numpy array): Detail coefficient (cH, cV, or cD).
        method (str): Type of augmentation ('noise', 'enhance', or 'mask').
        intensity (float): Intensity of the augmentation effect.

    Output:
        numpy array: Augmented detail coefficient.
    """
    if method == 'noise':
        noise = np.random.normal(loc=0, scale=intensity, size=coeff.shape)
        return coeff + noise
    elif method == 'enhance':
        return coeff * (1 + intensity)
    elif method == 'mask':
        mask = np.random.binomial(1, 1-intensity, size=coeff.shape)
        return coeff * mask
    else:
        return coeff

In [22]:
def decompose_and_inverse(image, wavelet='bior6.8'):
    """
    Performs wavelet decomposition on an image, applies augmentation to wavelet coefficients,
    and reconstructs the image from modified coefficients.

    Inputs:
    - image (torch.Tensor): A multi-channel image tensor.
    - wavelet (str): The type of wavelet to use for decomposition and inverse operation.

    Output:
    - coeffs_tensor (torch.Tensor): The reconstructed image tensor after wavelet modifications.
    """

    # Move the image to CPU if it's on GPU because pywt (PyWavelets) does not support CUDA tensors.
    if image.is_cuda:
        image = image.cpu()

    # Convert the tensor to a NumPy array for processing with pywt.
    image_np = image.numpy()

    # Define the target size for the final image tensor.
    target_size = (224, 224)
    
    # List to hold wavelet coefficients of each channel.
    coeffs_list = []

    # Process each channel in the image.
    for i in range(image.shape[0]):
        # Decompose the image using 2D Discrete Wavelet Transform.
        cA, (cH, cV, cD) = pywt.dwt2(image_np[i], wavelet)

        # Apply different augmentations to the high-frequency coefficients.
        cH = augment_details(cH, method='noise', intensity=np.random.uniform(0.05, 0.15))
        cV = augment_details(cV, method='enhance', intensity=np.random.uniform(0.1, 0.3))
        cD = augment_details(cD, method='mask', intensity=np.random.uniform(0.15, 0.25))

        # Reconstruct the image from the modified coefficients.
        coeffs_modified = (cA, (cH, cV, cD))
        image_reconstructed = pywt.idwt2(coeffs_modified, wavelet).astype(np.float32)

        # Convert the numpy array back to a tensor.
        coeffs_list.append(T.ToTensor()(image_reconstructed).squeeze())

    # Stack the tensors of all channels.
    coeffs_tensor = torch.stack(coeffs_list, dim=0)

    # Resize the tensor if it's not the target size.
    if coeffs_tensor.shape[1:] != target_size:
        coeffs_tensor = F.interpolate(coeffs_tensor.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)

    return coeffs_tensor

In [23]:
def only_decompose(image, wavelet='bior6.8'):
    """
    Perform Wavelet decomposition on an image, resize it to target size, and convert it back to PyTorch tensor.
    
    Inputs:
        image (torch.Tensor): Input image in the shape (C, H, W).
        wavelet (str): Name of the wavelet function to use.
    
    Output:
        torch.Tensor: Resized wavelet-decomposed image as a PyTorch tensor.
    """

    # Check if the image tensor is on the GPU (CUDA). Since the pywt library does not support CUDA tensors,
    # move the image to the CPU. This ensures compatibility with the pywt functions, which require
    # the image to be in a NumPy array format.
    if image.is_cuda:
        image = image.cpu()

    # Convert the image tensor to a numpy array.
    image_np = image.numpy()
    
    # Define the target size for the resized images.
    target_size = (224, 224)

    # Initialize a list to store the tensors of decomposed channels.
    coeffs_list = []

    # Loop through each channel in the image tensor.
    for i in range(image.shape[0]):
        # Perform the discrete wavelet transform (DWT) on the current channel.
        cA, (cH, cV, cD) = pywt.dwt2(image_np[i], wavelet)
        
        # Convert the approximation coefficients to a PIL image, scale to 0-255.
        cA_pil = Image.fromarray((cA * 255).astype(np.uint8))
        
        # Resize the image to the target size using bilinear interpolation.
        cA_resized = cA_pil.resize(target_size, Image.BILINEAR)
        
        # Convert the resized PIL image back to a PyTorch tensor.
        cA_tensor = T.ToTensor()(cA_resized)
        
        # Append the tensor to the list.
        coeffs_list.append(cA_tensor)

    # Stack all channel tensors along the first dimension to form a single tensor.
    coeffs_tensor = torch.stack(coeffs_list, dim=0).to(device)
    
    # Return the tensor of resized wavelet-decomposed images.
    return coeffs_tensor

In [24]:
def multilevel_decompose(image, wavelet='bior6.8', max_lev=3):
    """
    Performs a multilevel wavelet decomposition on an image, modifies the detail coefficients at each level,
    and progressively reconstructs the image to its original dimension.

    Inputs:
        image (torch.Tensor): The input image tensor in the shape (C, H, W).
        wavelet (str): The type of wavelet to use for decomposition.
        max_lev (int): Maximum number of decomposition levels.

    Output:
        torch.Tensor: The reconstructed image tensor after multilevel wavelet modifications.
    """

    # Ensure the image tensor is on the CPU, as PyWavelets does not support GPU tensors.
    if image.is_cuda:
        image = image.cpu()

    # Convert the image tensor to a numpy array.
    image_np = image.numpy()
    
    # Define the target size for the final image tensor.
    target_size = (224, 224)
    
    # List to hold the final reconstructed tensors for each channel.
    coeffs_list = []

    # Loop through each channel in the image tensor.
    for i in range(image.shape[0]):
        coeffs = []
        current_image = image_np[i]

        # Perform multilevel decomposition.
        for _ in range(max_lev):
            # Decompose the image using 2D Discrete Wavelet Transform.
            cA, (cH, cV, cD) = pywt.dwt2(current_image, wavelet)
            
            # Apply modifications to the high-frequency detail coefficients.
            cH = augment_details(cH, method='noise', intensity=np.random.uniform(0.05, 0.15))
            cV = augment_details(cV, method='enhance', intensity=np.random.uniform(0.1, 0.3))
            cD = augment_details(cD, method='mask', intensity=np.random.uniform(0.15, 0.25))
            
            # Store the current level's coefficients.
            coeffs.append((cA, (cH, cV, cD)))
            
            # Use the approximation coefficients as the input for the next level of decomposition.
            current_image = cA

        # Reconstruct the image from the deepest level back to the original.
        for level in range(max_lev - 1, -1, -1):
            cA, (cH, cV, cD) = coeffs[level]
            current_image = pywt.idwt2((cA, (cH, cV, cD)), wavelet).astype(np.float32)

        # Convert the reconstructed numpy array back to a tensor.
        reconstructed_tensor = T.ToTensor()(current_image).squeeze()
        coeffs_list.append(reconstructed_tensor)

    # Stack all reconstructed channel tensors.
    coeffs_tensor = torch.stack(coeffs_list, dim=0)

    # Resize the tensor to the target size if necessary.
    if coeffs_tensor.shape[1:] != target_size:
        coeffs_tensor = F.interpolate(coeffs_tensor.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)

    # Return the final reconstructed image tensor.
    return coeffs_tensor

The following functions reports various tests of assembling Wavelet Decomposition with Augmix.

- **augment_with_wavelets_and_augmix**: for each image performs the combination of wavelet and augmix M times, creating a batch of M augmented images;
- **augment_with_augmix_and_wavelets**: this function performs the same procedure as the previous one, inverting the order of wavelet and augmix;
- **augment_wavelet_and_M_augmix**: this function performs one wavelet decomposition and with the same decomposition creates a batch of M augmented images;
- **augment_M_augmix_and_wavelet**: on the other hand this function creates a batch of M augmented images and then decompose each image in batch.

We select a batch of 8 different wavelet functions, in first two functions, we use each wavelet function in the batch in a different order each time. In the last two functions we only use one wavelet function, uniformly sampled from the batch.

In [25]:
def augment_with_wavelets_and_augmix(single_image, wavelets_batch, decompose_function, args):
    """
    Inputs:
        single_image (torch.Tensor): The input image tensor with dimensions (C, H, W), where C is the 
                                     number of channels, H is height, and W is width.
        wavelets_batch (list): A list of wavelet names to be used for decomposition.
        decompose_function (function): The function that applies the wavelet decomposition to the image.

    Outputs:
        list of torch.Tensor: A list of augmented images after applying wavelet decomposition and 
                              further augmentations.
    """
    # Determine the number of wavelets available in the batch
    num_wavelets = len(wavelets_batch)
    
    # Decompose the image using each wavelet from the batch, cycling through wavelets if necessary
    decomposed_single_image = [
        decompose_function(single_image, wavelets_batch[i % num_wavelets]) for i in range(args.M)
    ]
    
    # Apply the AugMix augmentation strategy to each decomposed image to further enhance diversity
    augmented_images = [augmix(dec) for dec in decomposed_single_image]

    # Return the list of augmented images
    return augmented_images

In [26]:
def augment_with_augmix_and_wavelets(single_image, wavelets_batch, decompose_function, args):
    """
    Inputs:
        single_image (torch.Tensor): Input image tensor with dimensions (C, H, W), where C is the 
                                     number of channels, H is height, and W is width.
        wavelets_batch (list): A list of wavelet types to use for decomposition.
        decompose_function (function): Function to apply wavelet decomposition to the image.

    Outputs:
        list of torch.Tensor: A list of decomposed images after applying AugMix augmentation and 
                              wavelet decomposition.
    """

    # Number of wavelets available in the batch
    num_wavelets = len(wavelets_batch)

    # Apply AugMix augmentation to the image multiple times
    augmented_images = [augmix(single_image) for _ in range(args.M)]

    # Decompose each augmented image using wavelets, cycling through wavelets if necessary
    decomposed_images = [
        decompose_function(augmented_images[i], wavelets_batch[i % num_wavelets]) for i in range(args.M)
    ]

    # Return the list of decomposed images
    return decomposed_images

In [27]:
def augment_wavelet_and_M_augmix(single_image, wavelets_batch, decompose_function, args):
    """
    Inputs:
        single_image (torch.Tensor): The input image tensor in shape (C, H, W), where C is the 
                                     number of channels, H is height, and W is width.
        wavelets_batch (list): A list of wavelet types available for decomposition.
        decompose_function (function): The function that applies the wavelet decomposition to the image.

    Outputs:
        list of torch.Tensor: A list of AugMix-augmented images generated from a single wavelet-decomposed image.
    """

    # Ensure the image is moved to the appropriate computing device
    single_image = single_image.to(device)
    
    # Select a random wavelet from the batch
    random_wavelet = np.random.choice(wavelets_batch)
    
    # Decompose the image using the selected wavelet
    decomposed_single_image = decompose_function(single_image, wavelet=random_wavelet)

    # Generate a batch of M augmented images from the decomposed image
    augmented_images = [augmix(decomposed_single_image) for _ in range(args.M)]

    # Return the list of augmented images
    return augmented_images

In [28]:
def augment_M_augmix_and_wavelet(single_image, wavelets_batch, decompose_function, args):
    """
    Inputs:
        single_image (torch.Tensor): The input image tensor in shape (C, H, W), where C is the number of channels,
                                     H is height, and W is width.
        wavelets_batch (list): A list of wavelet types available for decomposition.
        decompose_function (function): The function that applies wavelet decomposition to the image.

    Outputs:
        list of torch.Tensor: A list of wavelet-decomposed images generated from each AugMix-augmented image.
    """

    # Move the original image to the specified device
    single_image = single_image.to(device)
    
    # Select a random wavelet from the provided batch
    random_wavelet = np.random.choice(wavelets_batch)
    
    # Generate a batch of M augmented images from the original image using AugMix
    augmented_images = [augmix(single_image) for _ in range(args.M)]

    # Apply wavelet decomposition to each augmented image using the selected random wavelet
    decomposed_single_images = [decompose_function(aug_img, wavelet=random_wavelet) for aug_img in augmented_images]

    # Return the list of decomposed images
    return decomposed_single_images

The next function has been created to handle the different augmentation functions during the tests.

In [29]:
def adapt_wavelet(single_image, augmentation_function, decompose_function, args):
    """
    Input:
        single_image (torch.Tensor): The input image tensor.
        augmentation_function (function): The function that applies wavelet decomposition and augmentation.
        decompose_function (function): The wavelet decomposition function.

    Output:
        torch.Tensor: The averaged output distribution from the model across different wavelet transformations.
    """

    # Ensure the image is on the correct device
    single_image = single_image.to(device)

    # List of wavelets to be used for augmentation
    wavelets_batch = ['haar', 'db7', 'bior6.8', 'sym4', 'coif3', 'sym8', 'db4', 'bior1.5']
    # Randomize the order of wavelets to introduce variability
    random.shuffle(wavelets_batch)

    # Apply wavelet decomposition and augmentation to the single image across different wavelet types
    augmented_images = augmentation_function(single_image, wavelets_batch, decompose_function, args)

    # Stack all augmented images into a tensor and ensure it's on the correct device
    augmented_images = torch.stack(augmented_images, dim=0).to(device)
    
    # Process the augmented images through the model to obtain outputs
    outputs = model(augmented_images)

    # Apply specific modifications if adversarial corruption is noted in args
    if args.corruption == 'adversarial':
        outputs = outputs[:, imagenet_a_mask]

    # Average the output distributions to get a final single distribution representing the model's response
    avg_out_distr = torch.mean(outputs, dim=0)

    # Return the averaged output distribution
    return avg_out_distr

And finally we can find a modified version of the MEMO test function.

In [30]:
def test_with_wavelet(test_loader, augmentation_function, decompose_function, model, args, device):
    """
    Test function
    
    Inputs:
    - test_loader (DataLoader): DataLoader containing the dataset to be tested, providing batches of images and labels.
    - augmentation_function: The sequence of augumentations to use
    - decompose_function: the type of decomposition to apply on the image
    - model (torch.nn.Module): The neural network model to be evaluated.
    - device (torch.device or str): The computing device (CPU or GPU) where the model is run.

    
    Outputs:
    - final_accuracy_top1 (float): Final top-1 accuracy percentage across the test set.
    - final_accuracy_top5 (float): Final top-5 accuracy percentage across the test set.
    """
    
    # Initial print to indicate the start of the testing phase
    print('Inside Test')
    
    # Initialize counters for samples processed and cumulative accuracies for top-1 and top-5
    samples = 0.0
    cumulative_accuracy_top1 = 0.0
    cumulative_accuracy_top5 = 0.0

    # Store the default state of the model parameters to restore later
    default_params = {}
    for param in model.parameters():
        default_params[param] = param.clone()
    
    # Iterate over each batch of images and labels in the test loader
    for i, (image_batch, label_batch) in enumerate(test_loader):
        
        # Move the image and label batch to the specified device
        image_batch, label_batch = image_batch.to(device), label_batch.to(device)

        # Set model to evaluation mode
        model.eval()
        
        # Perform augmentation and update model parameters multiple times per image
        for _ in range(args.Niter):
            # Reset gradient information
            optimizer.zero_grad()
            
            # Augment each image in the batch and calculate the average output distribution
            avg_out_distr = [adapt_wavelet(single_image, augmentation_function, decompose_function, args) for single_image in image_batch]
            avg_out_distr = torch.stack(avg_out_distr, dim=0)

            # Calculate the loss from the average output distributions
            loss = marginal_entropy(avg_out_distr)
            loss.backward()
            optimizer.step()

        # Disable gradient calculations for performance
        with torch.no_grad():
            # Forward pass to get model outputs
            output = model(image_batch)

            # Apply mask if corruption is adversarial
            if args.corruption == 'adversarial':
                output = output[:, imagenet_a_mask]

            # Calculate top-1 accuracy
            _, predicted_top1 = output.max(1)
            cumulative_accuracy_top1 += predicted_top1.eq(label_batch).sum().item()
        
            # Calculate top-5 accuracy
            _, predicted_top5 = output.topk(5, dim=1)
            correct_top5 = predicted_top5.eq(label_batch.view(-1, 1).expand_as(predicted_top5))
            cumulative_accuracy_top5 += correct_top5.sum().item()

            # Update total samples processed
            samples += image_batch.size(0)

            # Print current accuracy every 10 images
            if (i + 1) % 10 == 0:
                current_accuracy_top1 = cumulative_accuracy_top1 / samples * 100
                current_accuracy_top5 = cumulative_accuracy_top5 / samples * 100
                print(f'Image {i + 1}, Current Accuracy Top-1: {current_accuracy_top1:.4f}')
                print(f'Image {i + 1}, Current Accuracy Top-5: {current_accuracy_top5:.4f}')
            
        # Restore the original parameters after each batch to avoid parameter drift
        for param in model.parameters():
            param.data = default_params[param].clone()

    # Calculate and return the final accuracies for the entire test set
    final_accuracy_top1 = cumulative_accuracy_top1 / samples * 100
    final_accuracy_top5 = cumulative_accuracy_top5 / samples * 100
    
    return final_accuracy_top1, final_accuracy_top5

### Final Test

#### Test 1_s:
In this test use the decomposition function **decompose_and_inverse** and the augmentation function **augment_with_wavelets_and_augmix**.

In [31]:
final_accuracy_top1, final_accuracy_top5 = test_with_wavelet(
    test_loader,
    augment_with_wavelets_and_augmix,
    decompose_and_inverse,
    model,
    args,
    device
)

print(f'Final Accuracy Top-1: {final_accuracy_top1:.4f}')
print(f'Final Accuracy Top-5: {final_accuracy_top5:.4f}')

Inside Test


/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Image 10, Current Accuracy Top-1: 21.2500
Image 10, Current Accuracy Top-5: 43.7500
Image 20, Current Accuracy Top-1: 10.6250
Image 20, Current Accuracy Top-5: 25.0000
Image 30, Current Accuracy Top-1: 7.5000
Image 30, Current Accuracy Top-5: 20.0000
Image 40, Current Accuracy Top-1: 5.9375
Image 40, Current Accuracy Top-5: 20.7812
Image 50, Current Accuracy Top-1: 4.7500
Image 50, Current Accuracy Top-5: 20.1250
Image 60, Current Accuracy Top-1: 3.9583
Image 60, Current Accuracy Top-5: 19.2708
Image 70, Current Accuracy Top-1: 3.9286
Image 70, Current Accuracy Top-5: 21.6071
Image 80, Current Accuracy Top-1: 4.0625
Image 80, Current Accuracy Top-5: 22.5000
Image 90, Current Accuracy Top-1: 3.9583
Image 90, Current Accuracy Top-5: 25.2083
Image 100, Current Accuracy Top-1: 3.5625
Image 100, Current Accuracy Top-5: 23.6250
Image 110, Current Accuracy Top-1: 3.3523
Image 110, Current Accuracy Top-5: 23.6364
Image 120, Current Accuracy Top-1: 3.6458
Image 120, Current Accuracy Top-5: 24.3

#### Test 1_m:
In this test use the decomposition function **multilevel_decompose** and the augmentation function **augment_with_wavelets_and_augmix**.

In [34]:
torch.cuda.empty_cache()
final_accuracy_top1, final_accuracy_top5 = test_with_wavelet(
    test_loader,
    augment_with_wavelets_and_augmix,
    multilevel_decompose,
    model,
    args,
    device
)
print(f'Final Accuracy Top-1: {final_accuracy_top1:.4f}')
print(f'Final Accuracy Top-5: {final_accuracy_top5:.4f}')

Inside Test
Image 10, Current Accuracy Top-1: 25.6250
Image 10, Current Accuracy Top-5: 46.8750
Image 20, Current Accuracy Top-1: 13.1250
Image 20, Current Accuracy Top-5: 25.6250
Image 30, Current Accuracy Top-1: 9.1667
Image 30, Current Accuracy Top-5: 20.6250
Image 40, Current Accuracy Top-1: 7.3438
Image 40, Current Accuracy Top-5: 22.6562
Image 50, Current Accuracy Top-1: 5.8750
Image 50, Current Accuracy Top-5: 21.6250
Image 60, Current Accuracy Top-1: 5.0000
Image 60, Current Accuracy Top-5: 21.0417
Image 70, Current Accuracy Top-1: 4.6429
Image 70, Current Accuracy Top-5: 21.6071
Image 80, Current Accuracy Top-1: 5.1562
Image 80, Current Accuracy Top-5: 23.3594
Image 90, Current Accuracy Top-1: 5.4861
Image 90, Current Accuracy Top-5: 25.5556
Image 100, Current Accuracy Top-1: 4.9375
Image 100, Current Accuracy Top-5: 24.6250
Image 110, Current Accuracy Top-1: 4.6591
Image 110, Current Accuracy Top-5: 25.1136
Image 120, Current Accuracy Top-1: 4.8958
Image 120, Current Accuracy

#### Test 2_s
In this test use the decomposition function **decompose_and_inverse** and the augmentation function **augment_wavelet_and_M_augmix**.

In [35]:
torch.cuda.empty_cache()
final_accuracy_top1, final_accuracy_top5 = test_with_wavelet(
    test_loader,
    augment_wavelet_and_M_augmix,
    decompose_and_inverse,
    model,
    args,
    device
)
print(f'Final Accuracy Top-1: {final_accuracy_top1:.4f}')
print(f'Final Accuracy Top-5: {final_accuracy_top5:.4f}')

Inside Test
Image 10, Current Accuracy Top-1: 23.1250
Image 10, Current Accuracy Top-5: 40.6250
Image 20, Current Accuracy Top-1: 11.5625
Image 20, Current Accuracy Top-5: 22.5000
Image 30, Current Accuracy Top-1: 7.9167
Image 30, Current Accuracy Top-5: 17.0833
Image 40, Current Accuracy Top-1: 6.4062
Image 40, Current Accuracy Top-5: 20.6250
Image 50, Current Accuracy Top-1: 5.1250
Image 50, Current Accuracy Top-5: 19.5000
Image 60, Current Accuracy Top-1: 4.2708
Image 60, Current Accuracy Top-5: 18.7500
Image 70, Current Accuracy Top-1: 4.7321
Image 70, Current Accuracy Top-5: 21.6071
Image 80, Current Accuracy Top-1: 5.0781
Image 80, Current Accuracy Top-5: 23.1250
Image 90, Current Accuracy Top-1: 5.4167
Image 90, Current Accuracy Top-5: 24.7222
Image 100, Current Accuracy Top-1: 4.8750
Image 100, Current Accuracy Top-5: 23.8750
Image 110, Current Accuracy Top-1: 4.5455
Image 110, Current Accuracy Top-5: 23.6364
Image 120, Current Accuracy Top-1: 4.4271
Image 120, Current Accuracy

#### Test 2_m
In this test use the decomposition function **multilevel_decompose** and the augmentation function **augment_wavelet_and_M_augmix**.

In [36]:
torch.cuda.empty_cache()
final_accuracy_top1, final_accuracy_top5 = test_with_wavelet(
    test_loader,
    augment_wavelet_and_M_augmix,
    multilevel_decompose,
    model,
    args,
    device
)
print(f'Final Accuracy Top-1: {final_accuracy_top1:.4f}')
print(f'Final Accuracy Top-5: {final_accuracy_top5:.4f}')

Inside Test
Image 10, Current Accuracy Top-1: 21.8750
Image 10, Current Accuracy Top-5: 45.6250
Image 20, Current Accuracy Top-1: 10.9375
Image 20, Current Accuracy Top-5: 25.9375
Image 30, Current Accuracy Top-1: 7.2917
Image 30, Current Accuracy Top-5: 20.0000
Image 40, Current Accuracy Top-1: 6.0938
Image 40, Current Accuracy Top-5: 23.1250
Image 50, Current Accuracy Top-1: 4.8750
Image 50, Current Accuracy Top-5: 21.7500
Image 60, Current Accuracy Top-1: 4.1667
Image 60, Current Accuracy Top-5: 20.4167
Image 70, Current Accuracy Top-1: 4.3750
Image 70, Current Accuracy Top-5: 21.4286
Image 80, Current Accuracy Top-1: 4.6875
Image 80, Current Accuracy Top-5: 22.6562
Image 90, Current Accuracy Top-1: 4.7222
Image 90, Current Accuracy Top-5: 25.4167
Image 100, Current Accuracy Top-1: 4.2500
Image 100, Current Accuracy Top-5: 24.1875
Image 110, Current Accuracy Top-1: 3.9773
Image 110, Current Accuracy Top-5: 24.1477
Image 120, Current Accuracy Top-1: 4.4792
Image 120, Current Accuracy

## Results

The baseline model with v1 weights achieves a top-1 accuracy of **1.8533%** when the output labels are limited to those within ImageNet-A. We observe that the MEMO enhances the performance of the pre-trained model on ImageNet-A.
Further improvements are noted when MEMO is combined with other techniques.

In our tests Memo achieves **5.1867%** of top-1 accuracy enhancing the classification score of the baseline model ResNet-50.

Using the combination between MEMO and the Wavelet Decomposition of each image we have obtained several results:
- test 1s obtained a **5.0933%** of top-1 accuracy;
- test 1m obtained a **5.3867%** of top-1 accuracy;
- test 2s obtained a **5.7867%** of top-1 accuracy;
- test 2m obtained a **5.9067%** of top-1 accuracy;

Thanks to these tests we have noticed an increase in the accuracy of the model proportional to number of levels of the decomposition used. In addiction, models with a where Augmix is applied to the same decomposed image performs better (test 2).

### Extra-Test

We have also conducted some additional tests that we have not reported above. 

Firstly, the tests with the function **only_decompose** do not provide significant results, probably due to the fact that the approximated image was too far from the orignal one.

Secondly, we have also done some tests with **augment_M_augmix_and_wavelet** and **augment_with_augmix_and_wavelets** but we have seen a dicrease in the performances with respect to the version where the wavelet was computed before the Augmix.

# Conclusions

We demonstrated that MEMO (Marginal Entropy Minimization Optimization) is an effective technique for Test-Time Adaptation (TTA). By integrating Wavelet Decomposition with MEMO, we adopted a novel frequency-based approach to data augmentation. Initially, we were optimistic that this method would significantly enhance our results. Although some tests showed improvements over the standard MEMO application, the overall outcomes did not meet our expectations. We anticipated that decomposing the image in the frequency domain would provide diverse perspectives of the image, thereby boosting model performance. However, the enhancements were less pronounced than we hoped.

### Future Works

Possible advances in this field can be:
- the study of the optimal level of the multiresolution decomposition;
- the study of the action of different wavelet functions;
- the tuning of the parameters of the coefficient detail augmentations (cH, cV, cD);
- the test of the model with different methods for augmenting coefficient details (noise, mask, enhance);
- the introduction of the adaptive batch normalization as reported in the original paper of MEMO.

We have also expect an increase in the accuracy of the model as the number of augmentations (M) increase. 

# References

1. Wang, H., Ge, S., Xing, E. P., & Lipton, Z. C. (2021). [MEMO: Test Time Robustness via Adaptation and Augmentation](https://arxiv.org/pdf/2110.09506)
2. Y Shimizu, Z Zhang, R Batres. (2007) [The Wavelet Transform in Signal and Image Processing](https://link.springer.com/chapter/10.1007/978-1-84628-955-2_5#citeas)
3. Dan Hendrycks, Norman Mu, Ekin D. Cubuk, Barret Zoph, Justin Gilmer, Balaji Lakshminarayanan (2020) [AugMix: A Simple Data Processing Method to Improve Robustness and Uncertainty](https://arxiv.org/abs/1912.02781)